In [60]:
# === NOTEBOOK 03: FINAL SUMMARY v3 ===
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, desc, sum
import pandas as pd

spark = SparkSession.builder.getOrCreate()

print("="*70)
print(" NOTEBOOK 03: FINAL SUMMARY & CONCLUSION v3")
print("="*70)

# Load data
df = spark.read.parquet("/lakehouse/default/Files/umkm_v3_final")

print(f"\n✅ Total UMKM: {df.count()}")

# 1. Ringkasan Statistik
print("\n" + "="*70)
print(" RINGKASAN STATISTIK")
print("="*70)

print(f"Skor Potensi     : Min {df.agg({'skor_potensi': 'min'}).collect()[0][0]} | Max {df.agg({'skor_potensi': 'max'}).collect()[0][0]} | Avg {df.agg({'skor_potensi': 'avg'}).collect()[0][0]:.1f}")
print(f"Omset Bulanan    : Avg Rp {df.agg({'omset_bulanan': 'avg'}).collect()[0][0]:,.0f}")
print(f"Persaingan Index : Avg {df.agg({'persaingan_index': 'avg'}).collect()[0][0]:.2f}")
print(f"Risiko Banjir    : Avg {df.agg({'risiko_banjir': 'avg'}).collect()[0][0]:.2f}")

# 2. Distribusi per Jenis Usaha
print("\n>>> DISTRIBUSI PER JENIS USAHA")
df.groupBy("jenis_usaha").agg(count("*").alias("jumlah")).orderBy(desc("jumlah")).show()

# 3. Top 5 Rekomendasi per Jenis Usaha (Ringkas)
print("\n" + "="*70)
print(" TOP 5 REKOMENDASI PER JENIS USAHA")
print("="*70)

for jenis in ["Makanan", "Fashion", "Kerajinan", "Jasa", "Pertanian"]:
    top5 = df.filter(col("jenis_usaha") == jenis) \
             .orderBy(col("skor_potensi").desc()) \
             .select("kabupaten", "kecamatan", "skor_potensi") \
             .limit(5).collect()
    
    print(f"\n>>> {jenis.upper()}")
    for i, row in enumerate(top5, 1):
        print(f"   {i}. {row['kecamatan']}, {row['kabupaten']} (Skor: {row['skor_potensi']:.1f})")

# 4. 5 Kecamatan Prioritas Tertinggi + Action Plan
print("\n" + "="*70)
print(" TOP 5 KECAMATAN PRIORITAS PEMERINTAH + ACTION PLAN")
print("="*70)

priority = df.groupBy("kabupaten", "kecamatan") \
    .agg(
        avg("skor_potensi").alias("avg_score"),
        count("*").alias("jumlah_umkm"),
        avg("persaingan_index").alias("avg_competition"),
        avg("risiko_banjir").alias("avg_risiko")
    ) \
    .orderBy(col("avg_score").asc()) \
    .limit(5).collect()

for i, row in enumerate(priority, 1):
    print(f"\n{i}. {row['kecamatan']}, {row['kabupaten']}")
    print(f"   Avg Skor      : {row['avg_score']:.1f}")
    print(f"   UMKM          : {row['jumlah_umkm']}")
    print(f"   Persaingan    : {row['avg_competition']:.2f}")
    print(f"   Risiko        : {row['avg_risiko']:.2f}")
    
    if row['avg_risiko'] > 0.6:
        print("   🎯 ACTION: Relokasi bantuan + KUR Mikro + asuransi UMKM")
    elif row['avg_competition'] > 0.7:
        print("   🎯 ACTION: Program diferensiasi + pelatihan branding + SEHATI")
    else:
        print("   🎯 ACTION: Paket lengkap (KUR + pelatihan + pendampingan)")

# 5. Estimasi Dampak
print("\n" + "="*70)
print(" ESTIMASI DAMPAK PROYEK")
print("="*70)

total_umkm = df.count()
print(f"Total UMKM di Jawa Barat (simulasi)     : {total_umkm}")
print(f"Potensi UMKM yang bisa diselamatkan    : ~{int(total_umkm * 0.22)} (22%)")
print(f"Potensi lapangan kerja terjaga         : ~{int(total_umkm * 0.22 * 2.3)} orang")
print(f"Potensi peningkatan omset kolektif     : ~Rp {int(total_umkm * 0.22 * 85000000):,}")

# 6. Kesimpulan untuk Portofolio
print("\n" + "="*70)
print(" KESIMPULAN UNTUK PORTFOLIO")
print("="*70)

print("""
✅ YANG BERHASIL DIBUAT:
• 6.000 data UMKM Jawa Barat dengan skor realistis (45-95)
• 8 Cluster Spasial dengan karakteristik unik
• Top 15 Rekomendasi Lokasi per jenis usaha
• What-If Analysis Simulator
• Government Priority Dashboard (15 kecamatan prioritas)
• Peta Interaktif dengan 3 kategori skor

🎯 DAMPAK:
• Membantu UMKM pilih lokasi dengan potensi lebih tinggi
• Membantu pemerintah target program KUR/SEHATI lebih tepat
• Mendukung perencanaan Smart City di Jawa Barat

📌 CATATAN:
• Data yang digunakan adalah data sintetis untuk proof-of-concept
• Di implementasi nyata, gunakan data BPS + Kemenkop UKM real-time
• Skor bervariasi 45-95 mencerminkan kondisi nyata

🚀 NEXT STEP:
• Integrasi data real-time BPS & Kemenkop UKM
• Mobile App untuk UMKM
• API untuk Bank/Fintech (credit scoring berbasis lokasi)
• Ekspansi ke Jawa Timur, Jawa Tengah, Sumatera
""")

print("="*70)
print(" PROYEK SELESAI — SIAP UNTUK PORTFOLIO & PRESENTASI")
print("="*70)

StatementMeta(yudhaesparkpool, 2, 67, Finished, Available, Finished, False)

 NOTEBOOK 03: FINAL SUMMARY & CONCLUSION v3

✅ Total UMKM: 6000

 RINGKASAN STATISTIK
Skor Potensi     : Min 45.0 | Max 90.0 | Avg 65.2
Omset Bulanan    : Avg Rp 93,943,822
Persaingan Index : Avg 0.67
Risiko Banjir    : Avg 0.61

>>> DISTRIBUSI PER JENIS USAHA
+-----------+------+
|jenis_usaha|jumlah|
+-----------+------+
|  Pertanian|  1230|
|    Makanan|  1223|
|       Jasa|  1199|
|  Kerajinan|  1175|
|    Fashion|  1173|
+-----------+------+


 TOP 5 REKOMENDASI PER JENIS USAHA

>>> MAKANAN
   1. Kecamatan 4, Purwakarta (Skor: 90.0)
   2. Kecamatan 24, Bandung (Skor: 89.0)
   3. Kecamatan 28, Purwakarta (Skor: 89.0)
   4. Kecamatan 1, Banjar (Skor: 88.0)
   5. Kecamatan 12, Karawang (Skor: 88.0)

>>> FASHION
   1. Kecamatan 27, Subang (Skor: 90.0)
   2. Kecamatan 9, Tasikmalaya (Skor: 89.0)
   3. Kecamatan 2, Banjar (Skor: 89.0)
   4. Kecamatan 13, Purwakarta (Skor: 89.0)
   5. Kecamatan 33, Banjar (Skor: 89.0)

>>> KERAJINAN
   1. Kecamatan 2, Ciamis (Skor: 90.0)
   2. Kecamatan 1